In [1]:
%pip install -U langchain langchain-groq langchain-community requests duckduckgo-search python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import requests
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing. Add it to your .env file.")
if not WEATHERSTACK_API_KEY:
    raise ValueError("WEATHERSTACK_API_KEY is missing. Add it to your .env file.")

C:\Users\aksha\AppData\Local\Temp\ipykernel_30784\3993084930.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [3]:
# Tool 1: DuckDuckGo web search
search_tool = DuckDuckGoSearchRun()

print("DuckDuckGo search tool is ready.")

DuckDuckGo search tool is ready.


In [ ]:
# Tool 2: Get current weather using WeatherStack
@tool
def get_weather_data(city: str) -> str:
    """Fetch current weather data for the given city."""
    api_key = os.getenv("WEATHERSTACK_API_KEY") or WEATHERSTACK_API_KEY
    url = "http://api.weatherstack.com/current"
    params = {
        "access_key": api_key,
        "query": city
    }

    response = requests.get(url, params=params, timeout=20)
    response.raise_for_status()
    data = response.json()

    if "error" in data:
        return f"WeatherStack API error: {data['error'].get('info', 'Unknown error')}"

    current = data.get("current", {})
    location = data.get("location", {})

    return (
        f"Weather in {location.get('name', city)}, "
        f"{location.get('country', '')}: "
        f"{current.get('temperature', 'N/A')}°C, "
        f"{', '.join(current.get('weather_descriptions', []))}. "
        f"Feels like {current.get('feelslike', 'N/A')}°C. "
        f"Humidity: {current.get('humidity', 'N/A')}%. "
    )

print("Weather tool is ready.")


In [8]:
# Initialize LLM and Agent with tools
import os
import requests
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

# Ensure tools are available even if previous cells were skipped
try:
    search_tool
except NameError:
    search_tool = DuckDuckGoSearchRun()

try:
    get_weather_data
except NameError:
    @tool
    def get_weather_data(city: str) -> str:
        """Fetch current weather data for the given city."""
        api_key = os.getenv("WEATHERSTACK_API_KEY")
        url = "http://api.weatherstack.com/current"
        params = {
            "access_key": api_key,
            "query": city
        }
        response = requests.get(url, params=params, timeout=20)
        response.raise_for_status()
        data = response.json()
        if "error" in data:
            return f"WeatherStack API error: {data['error'].get('info', 'Unknown error')}"
        current = data.get("current", {})
        location = data.get("location", {})
        return (
            f"Weather in {location.get('name', city)}, "
            f"{location.get('country', '')}: "
            f"{current.get('temperature', 'N/A')}°C, "
            f"{', '.join(current.get('weather_descriptions', []))}. "
            f"Feels like {current.get('feelslike', 'N/A')}°C. "
            f"Humidity: {current.get('humidity', 'N/A')}%. "
        )

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

tools = [search_tool, get_weather_data]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant with access to web search and weather tools."
)

# Test the agent
response = agent.invoke({
    "messages": [HumanMessage("What is the current weather in Mumbai?")]
})
print(response["messages"][-1].content)


The latest weather report for **Mumbai, India** shows:

- **Temperature:** 29 °C  
- **Feels like:** 32 °C  
- **Condition:** Smoky haze  
- **Humidity:** 66 %

So it’s warm and a bit hazy right now, with moderate humidity. Stay hydrated if you’re heading outdoors!


In [9]:
# Initialize Groq chat model
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

print("Groq model is ready.")

Groq model is ready.


In [10]:
from langchain.agents import create_agent

tools = [search_tool, get_weather_data]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful assistant. Use the web search tool for general "
        "information and the weather tool for current weather. "
        "When a question requires both, use both tools before answering. "
        "Clearly distinguish retrieved facts from uncertainty."
    )
)

print("Agent created successfully.")

Agent created successfully.


In [11]:
# Invoke the agent
user_query = (
    "Find the capital of Madhya Pradesh, then find its current weather condition."
)

response = agent.invoke({
    "messages": [{"role": "user", "content": user_query}]
})

print(response["messages"][-1].content)

**Capital of Madhya Pradesh:**  
- The capital city is **Bhopal**【duckduckgo_search】.

**Current weather in Bhopal (as of the latest data):**  
- Temperature: **22 °C**  
- Condition: **Clear**  
- Feels like: **20 °C**  
- Humidity: **36 %**【get_weather_data】

*All facts are taken from the retrieved sources; no additional assumptions were made.*


In [12]:
# Optional: inspect the complete conversation, including tool calls
for message in response["messages"]:
    print(f"\\n--- {message.type} ---")
    print(message.content)

    # Tool calls are attached to assistant messages when the model requests tools.
    if getattr(message, "tool_calls", None):
        print("Tool calls:", message.tool_calls)

\n--- human ---
Find the capital of Madhya Pradesh, then find its current weather condition.
\n--- ai ---

Tool calls: [{'name': 'duckduckgo_search', 'args': {'query': 'capital of Madhya Pradesh'}, 'id': 'fc_1282d51c-64f3-4eb7-b762-e8ddbb31dfb9', 'type': 'tool_call'}]
\n--- tool ---
Its capital is Bhopal. Other major cities include Indore, Gwalior, Jabalpur, Chhindwara and Sagar. Madhya Pradesh is the second largest Indian state by area and the fifth largest state by population with over 72 million residents. Bhopal (Hindi: Bhōpāl, pronounced [bʱoːpaːl] ⓘ) is the capital city of the Indian state of Madhya Pradesh and the administrative headquarters of both Bhopal district and Bhopal division. [10] . It is known as the City of Lakes, [11] due to presence of various natural and artificial lakes near the city boundary. The capital of Madhya Pradesh, Bhopal, is a city of administrative power, cultural richness, and natural beauty. Its central location, historical importance, and well-devel